# ⚡ Realtime WebRTC Voice Agent (Kaggle GPU + Cloudflare Tunnel)

This notebook runs the **Full-Duplex WebRTC Voice Agent Server** on a Kaggle GPU (T4 / P100 / A100):
- 🎙️ **ASR**: Qwen3-ASR (0.6B) running in **FP16 CUDA** (~15ms chunk, <200ms decode)
- 🧠 **VAD**: Silero Neural VAD with adaptive voice energy boost
- 💬 **LLM**: Local GPU LLaMA.cpp Gemma 4 E2B Instruct INT8 GGUF (>100 tokens/sec, sub-50ms TTFT)
- 🔊 **TTS**: Kokoro-82M Realtime Neural GPU (<30ms latency, Hindi hf_alpha & English af_heart)
- 🌐 **WebRTC**: Real-time peer-to-peer audio streaming & DataChannel telemetry over **Cloudflare Quick Tunnel**

---

In [ ]:
# 1. Check GPU Status
!nvidia-smi

In [ ]:
# 2. Clone Repository & Navigate
!git clone https://github.com/yashNiwane/realtime-voice-agent-webrtc.git /kaggle/working/realtime-voice-agent-webrtc
%cd /kaggle/working/realtime-voice-agent-webrtc

In [ ]:
# 3. Install System libsrtp2, espeak-ng & Compile Python Dependencies (OpenSSL 3 + llama-cpp CUDA + Kokoro-82M)
!apt-get update -qq && apt-get install -y -qq libsrtp2-dev espeak-ng libsndfile1 pkg-config wget curl
!pip install --quiet --no-binary pylibsrtp --no-cache-dir pylibsrtp
!pip install --quiet llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 || pip install --quiet llama-cpp-python
!pip install --quiet kokoro soundfile misaki
!pip install --quiet -r requirements.txt

# Pre-download & warm Kokoro-82M neural models
!python -c "import pylibsrtp, aiortc, llama_cpp; from kokoro import KPipeline; print('📥 Downloading Kokoro models...'); KPipeline(lang_code='h'); KPipeline(lang_code='a'); print('✅ Kokoro-82M downloaded & warm on GPU!')"

In [ ]:
# 4. Launch Cloudflare Tunnel & Print Public WebRTC Agent URL
import subprocess, time, re, os

if not os.path.exists("cloudflared"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# Start tunnel
!rm -f /tmp/tunnel.log
tunnel_proc = subprocess.Popen("./cloudflared tunnel --url http://localhost:7860 > /tmp/tunnel.log 2>&1", shell=True)

tunnel_url = None
for _ in range(25):
    time.sleep(1)
    if os.path.exists("/tmp/tunnel.log"):
        with open("/tmp/tunnel.log", "r") as f:
            content = f.read()
            matches = re.findall(r"https://[-0-9a-z]*\.trycloudflare\.com", content)
            if matches:
                tunnel_url = matches[-1]
                break

print("=" * 70)
if tunnel_url:
    print(f"🚀 PUBLIC WEBRTC DASHBOARD URL: {tunnel_url}")
    print(f"📡 REST SIGNALING ENDPOINT:     {tunnel_url}/offer")
else:
    print("⚠️ Tunnel initializing. Check /tmp/tunnel.log")
print("=" * 70)

In [ ]:
# 5. Start Full-Duplex WebRTC Voice Agent Server (GPU Accelerated Kokoro & Gemma INT8)
os.environ["DEVICE"] = "cuda"
os.environ["TORCH_DTYPE"] = "float16"
os.environ["PORT"] = "7860"
os.environ["HOST"] = "0.0.0.0"
os.environ["LLM_ENGINE_TYPE"] = "llama_cpp"
os.environ["LLM_N_GPU_LAYERS"] = "-1"
os.environ["LLM_REPO_ID"] = "unsloth/gemma-4-E2B-it-GGUF"
os.environ["LLM_GGUF_FILENAME"] = "gemma-4-E2B-it-Q8_0.gguf"
os.environ["LLM_MODEL"] = "gemma-4-e2b-it-int8"
os.environ["TTS_ENGINE"] = "kokoro"
os.environ["KOKORO_VOICE_HI"] = "hf_alpha"
os.environ["KOKORO_VOICE_EN"] = "af_heart"
os.environ["KOKORO_SPEED"] = "1.05"
os.environ["PYTHONFAULTHANDLER"] = "1"

!python -m server.webrtc_server